# Dana's Bakery Order Desk

Use this beginner-friendly notebook to process orders for Dana's Bakery in Nachlaot. Enter a product, quantity, weekday, and city. A blank city means shop pickup. Inventory is shared across orders in this Colab runtime and resets when the runtime restarts.


## 1. Set up the menu and today's inventory

Prices and starting stock are stored in dictionaries so they are easy to update. The substitution path is babka → rugelach → sourdough → challah.


In [ ]:
# Current menu prices in shekels (₪). Croissants are intentionally not orderable.
MENU = {
    'challah': 18,
    'babka': 45,
    'box of rugelach': 30,
    'sourdough loaf': 25,
}

# Starting stock for today's orders.
STARTING_INVENTORY = {
    'challah': 10,
    'babka': 0,
    'box of rugelach': 3,
    'sourdough loaf': 4,
}

# Try the next item in this sequence when an item is out of stock.
REPLACEMENT = {
    'babka': 'box of rugelach',
    'box of rugelach': 'sourdough loaf',
    'sourdough loaf': 'challah',
}

WEEKDAYS = {'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'}
inventory = STARTING_INVENTORY.copy()  # This changes as orders are processed.
order_diaries = []  # Keep a copy of every order's step-by-step diary.


## 2. Add the order-processing function

The function validates each order, checks available stock, makes up to two replacement suggestions, calculates delivery and discounts, updates inventory, and prints a diary plus a warm customer message.


In [ ]:
def process_order(product, quantity, day, city=''):
    # Keep a clear record of every stage, including invalid or cancelled orders.
    diary = []
    diary.append(f'Order received: product={product!r}, quantity={quantity!r}, day={day!r}, city={city!r}.')

    def finish(status, message):
        diary.append(f'Customer response: {message}')
        order_diaries.append(diary.copy())
        print('ORDER DIARY')
        for step_number, step in enumerate(diary, start=1):
            print(f'{step_number}. {step}')
        return {'status': status, 'message': message, 'diary': diary.copy()}

    # Normalize product spelling and reject items outside the current menu.
    product_key = str(product).strip().casefold()
    menu_names = {name.casefold(): name for name in MENU}
    if product_key in {'croissant', 'croissants'}:
        diary.append('Validation: croissants were on an old menu but are discontinued.')
        return finish('unavailable', "Hi there! We no longer bake croissants, so I couldn't place that order. Please choose from challah, babka, a box of rugelach, or a sourdough loaf. —Dana")
    if product_key not in menu_names:
        diary.append(f'Validation failed: {product!r} is not on the current menu.')
        return finish('not understood', "Hi there! I couldn't understand that product. Please choose challah, babka, a box of rugelach, or a sourdough loaf. —Dana")
    requested_product = menu_names[product_key]

    # Quantity must be a whole number greater than zero.
    try:
        order_quantity = int(quantity)
        if str(quantity).strip() != str(order_quantity) or order_quantity <= 0:
            raise ValueError
    except (TypeError, ValueError):
        diary.append(f'Validation failed: {quantity!r} is not a positive whole-number quantity.')
        return finish('not understood', "Hi there! I couldn't understand the quantity. Please enter a whole number greater than zero, and I'll be happy to help. —Dana")

    order_day = str(day).strip().casefold()
    if order_day not in WEEKDAYS:
        diary.append(f'Validation failed: {day!r} is not a weekday name.')
        return finish('not understood', "Hi there! I couldn't understand the day. Please enter a weekday, such as Monday or Friday. —Dana")
    diary.append(f'Validation passed: {order_quantity} × {requested_product}, for {order_day.title()}.')

    # If we cannot fill the full quantity, try no more than two items in the fallback sequence.
    selected_product = requested_product
    replaced = False
    if inventory[requested_product] < order_quantity:
        diary.append(f'Stock check: only {inventory[requested_product]} {requested_product}(s) available; {order_quantity} requested.')
        next_product = requested_product
        for suggestion_number in range(1, 3):
            alternative = REPLACEMENT.get(next_product)
            if alternative is None:
                diary.append('Replacement check: no further item is listed in the replacement sequence.')
                break
            diary.append(f'Suggestion {suggestion_number}: offer {alternative} for the full quantity of {order_quantity}.')
            if inventory[alternative] >= order_quantity:
                selected_product = alternative
                replaced = True
                diary.append(f'Replacement selected: {inventory[alternative]} {alternative}(s) are available for the full quantity.')
                break
            diary.append(f'Suggestion {suggestion_number} cannot fulfill the full quantity; available stock is {inventory[alternative]}.')
            next_product = alternative
        if not replaced:
            diary.append('Order cancelled: no offered replacement could cover the full quantity within two suggestions.')
            return finish('cancelled', "I'm so sorry, but we're out of stock for that quantity and couldn't find a suitable replacement after two suggestions. I've cancelled the order. —Dana")
    else:
        diary.append(f'Stock check: {inventory[requested_product]} {requested_product}(s) available; order can be filled as requested.')

    # Only Jerusalem orders are delivered. Other cities are switched to free pickup.
    normalized_city = str(city).strip()
    delivery_fee = 0
    delivery_note = ''
    if not normalized_city:
        fulfillment = 'shop pickup'
        diary.append('Delivery: city was blank, so this order is free shop pickup.')
    elif normalized_city.casefold() == 'jerusalem':
        fulfillment = 'delivery in Jerusalem'
        delivery_fee = 20
        diary.append('Delivery: Jerusalem delivery selected; fee is ₪20.')
    else:
        fulfillment = 'free shop pickup'
        delivery_note = f'We do not deliver to {normalized_city}, so I switched this to free shop pickup.'
        diary.append(f'Delivery: {normalized_city} is outside our delivery area; changed to free pickup at no charge.')

    # Charge the replacement's price and discount baked goods only, never delivery.
    baked_goods_subtotal = MENU[selected_product] * order_quantity
    discount = 0
    if baked_goods_subtotal > 150 and order_day != 'friday':
        discount = round(baked_goods_subtotal * 0.15, 2)
        diary.append(f'Discount: 15% off baked goods over ₪150; discount is ₪{discount:.2f}.')
    elif baked_goods_subtotal > 150 and order_day == 'friday':
        diary.append('Discount: Friday order; the 15% baked-goods discount does not apply.')
    else:
        diary.append(f'Discount: baked-goods subtotal is ₪{baked_goods_subtotal}, so no discount applies.')
    total = round(baked_goods_subtotal - discount + delivery_fee, 2)

    inventory[selected_product] -= order_quantity
    diary.append(f'Inventory updated: {inventory[selected_product]} {selected_product}(s) remain.')
    diary.append(f'Price summary: goods ₪{baked_goods_subtotal}, discount ₪{discount:.2f}, delivery ₪{delivery_fee}, total ₪{total:.2f}.')

    replacement_note = (f" We were out of {requested_product}, so I substituted {selected_product} and kept your quantity at {order_quantity}." if replaced else '')
    message = f"Hi there! Thanks for ordering {order_quantity} {selected_product}(s) from Dana's Bakery.{replacement_note} {delivery_note} Your {fulfillment} order total is ₪{total:.2f}. We look forward to seeing you! —Dana"
    return finish('confirmed', message)


## 3. Enter and process an order

Run this cell whenever you want to enter an order. Rerunning only this cell keeps the current inventory; restarting the Colab runtime resets inventory to today's starting amounts.


In [ ]:
product = input('Product: ')
quantity = input('Quantity (whole number): ')
day = input('Day of the week: ')
city = input('Customer city (leave blank for shop pickup): ')

order_result = process_order(product, quantity, day, city)
print('\nCUSTOMER MESSAGE')
print(order_result['message'])


## Inventory at a glance

Run this optional cell to see remaining stock after processing orders.


In [ ]:
print('Current inventory:')
for item, amount in inventory.items():
    print(f'- {item}: {amount}')
